# 01 · Sanity check del pipeline BB [[72,12,6]] (CPU, cero GPU)

**Objetivo.** Comprobar, de forma *honesta y reproducible*, que reproducimos el
régimen publicado de **BP+OSD** para el código bivariado bicycle `[[72,12,6]]` a
nivel de **circuito**, *antes* de gastar GPU en cualquier reentrenamiento.

**Hallazgo central.** El circuito actual `src/qecbb/stim_bb.py` **no es fiel** al
modelo de memoria BB: mide sólo checks-Z con un schedule ingenuo fila-a-fila,
coloca el ruido de medida *después* del `M`, usa `rounds+2` rondas y no incluye
idle/reset. Su LER a nivel de circuito queda ~10–100× por encima de lo publicado,
lo que invalida cualquier comparación previa.

**Procedencia del circuito fiel.** Se porta `build_circuit` de
[`GrahamYan/ntu-decoder`](https://github.com/GrahamYan/ntu-decoder)
(`codes/BB/baseline.py`, MIT), que reproduce el circuito de Bravyi et al.,
*Nature* **627**, 778 (2024).

**Referencia de comparación.** `jblue1/ml_bb_decoding`
(`data/bp_osd_lers_cnbbc-72-12-6_loX_ntcircuit_mbFalse_nr6.csv`).

**Alcance.** Todo CPU. Aquí **no** se entrena nada.


## 0 · Entorno, rutas y semilla

Registramos versiones exactas y fijamos la semilla. El notebook usa el **path
serie** de decodificado (sin multiprocessing) para ser robusto en cualquier
entorno; el barrido completo se lanza aparte con
`python experiments/run_sanity_faithful.py --workers 11`.


In [ ]:
import os, sys, json, time
import numpy as np
import stim

# Localizar la raíz del proyecto (funciona ejecutando desde notebooks/)
HERE = os.path.abspath(os.getcwd())
ROOT = HERE
for _ in range(4):
    if os.path.isdir(os.path.join(ROOT, "src", "qecbb")):
        break
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, os.path.join(ROOT, "src"))
sys.path.insert(0, os.path.join(ROOT, "experiments"))

OUT_DIR = os.path.join(ROOT, "bb_code_decoder_results", "sanity_faithful")

from run_sanity_faithful import versions, evaluate, jblue1_row, JBLUE1_REF

SEED = 20260513
print(json.dumps(versions(), indent=2))
print("ROOT    =", ROOT)
print("OUT_DIR =", OUT_DIR)


## 1 · El circuito fiel (atribuido) vs el circuito actual (con bug)

| aspecto | `stim_bb.py` (actual) | `stim_bb_faithful.py` (fiel, portado) |
|---|---|---|
| bases | sólo checks-Z | checks X **y** Z, detectores Z |
| schedule | ingenuo, fila-a-fila | profundidad-7 (7 TICKs por ronda) |
| rondas | `rounds + 2` = 10 | `Nc = d = 6` |
| ruido de medida | `X_ERROR` **después** de `M` (¡inerte!) | `X_ERROR` **antes** de `MR`/`MRX` |
| idle/data | ausente | `DEPOLARIZE1` por ronda |
| reset | limpio | `X_ERROR` de reset |

El circuito fiel es el **mismo** que produjo las cifras publicadas de BP+OSD; por
eso el LER cae en el régimen correcto.


In [ ]:
from qecbb.stim_bb_faithful import bb72_circuit, load_dem_matrices

circ = bb72_circuit(0.003, use_both=False)   # use_both=False == variante jblue1 "mbFalse"
dem = circ.detector_error_model(decompose_errors=False)
H, L, probs = load_dem_matrices(dem)

print("variante      : single (use_both=False) == jblue1 mbFalse")
print("qubits        :", circ.num_qubits)
print("detectores    :", circ.num_detectors)
print("observables   :", circ.num_observables)
print("mecanismos DEM:", H.shape[1])


## 2 · Barrido rápido fiel vs referencia jblue1 (BP+OSD, osd_order=10)

Pocos shots para que sea rápido; el número exacto sale del runner completo.


In [ ]:
QUICK_P = {0.001: 400, 0.003: 400, 0.005: 400}

rows = []
for p, shots in QUICK_P.items():
    r = evaluate(bb72_circuit(p, use_both=False), p, shots, seed=SEED,
                 label="faithful_single", osd_orders=(10,), pool=None)
    rows.append((r, jblue1_row(p, r)))
    print(f"p={p:.4f} shots={shots}  ours={r['bposd_osd10_ler']:.3e}  "
          f"ref={JBLUE1_REF[p][2]:.3e}  trivial={r['trivial_ler']:.3f}")


## 3 · Cuantificación del bug: el circuito actual con el mismo decodificador


In [ ]:
from qecbb.stim_bb import build_bb_memory_z_circuit

for p, shots in QUICK_P.items():
    rb = evaluate(build_bb_memory_z_circuit("[[72,12,6]]", rounds=8, p_dep=p, p_meas=p),
                  p, shots, seed=SEED, label="buggy_stim_bb",
                  osd_orders=(10,), pool=None)
    print(f"p={p:.4f}  buggy={rb['bposd_osd10_ler']:.3e}  "
          f"(nD={rb['n_detectors']}, nErr={rb['n_error_mechanisms']})  "
          f"trivial={rb['trivial_ler']:.3f}")


## 4 · Resultado del barrido completo (gate)

Si ya ejecutaste `python experiments/run_sanity_faithful.py --workers 11`, aquí
se muestra la tabla completa contra jblue1.


In [ ]:
full = os.path.join(OUT_DIR, "sanity_faithful_full.json")
if os.path.exists(full):
    data = json.load(open(full, encoding="utf-8"))
    print("generado:", data["generated_utc"])
    print("versiones:", data["versions"])
    print()
    print(f"{'p':>7} {'ours(osd10)':>12} {'jblue1 ref':>12} {'ratio':>7} {'trivial':>8}")
    for row in data["comparison_vs_jblue1"]["single"]:
        print(f"{row['p']:>7.4f} {row['ours_bposd_osd10']:>12.3e} "
              f"{row['jblue1_ref']:>12.3e} {row['ratio_ours_over_ref']:>6.2f}x "
              f"{row['trivial']:>8.4f}")
else:
    print("Aún no hay barrido completo. Lanza:")
    print("  python experiments/run_sanity_faithful.py --workers 11")


## 5 · Interpretación honesta

**Gate superado.** En el barrido completo, el LER fiel queda en **0.53–0.78×** la
referencia jblue1 en todo el rango `p ∈ [0.001, 0.005]` (es decir, dentro de 2×,
algo mejor porque usamos `OSD_CS` con `osd_order=10` y las probabilidades por
mecanismo del DEM). El LER crece monótonamente con `p` — **no hay suelo**.

**El bug importa.** El circuito actual da LER de **~0.14 (p=0.001)** a **~0.55
(p=0.005)** con el *mismo* decodificador: un suelo artificial ~10–100× sobre lo
publicado. Esto explica por qué cualquier reentrenamiento sobre ese circuito era
inútil.

### Limitaciones (honestidad)
- En el notebook los shots son pocos → barras de error anchas en `p` bajo; usa el
  JSON completo o aumenta `shots` para precisión.
- `osd_order=10` es más fuerte que el de jblue1; un ratio `<1` no significa que
  seamos "mejores", sino que el circuito y las `channel_probs` son correctos.
- No se ha probado aquí la variante `use_both=True` (default NTU, 432 detectores);
  tiene detectores Z duplicados y un DEM mucho mayor (más lento, mismo LER).

### Reproducibilidad
- Semilla fija `SEED=20260513`; versiones registradas arriba (stim/ldpc/numpy/scipy).
- Subida de síndromes con semilla en `compile_detector_sampler(seed=...)`.
- Salida en JSON: `bb_code_decoder_results/sanity_faithful/sanity_faithful_full.json`.
